# ⚡ BPE Tokenizer — Sequential vs CUDA Efficiency Analysis
### Penn Treebank Dataset

This notebook **measures and compares** the performance of:
- 🐢 **Sequential** — single-threaded CPU merge loop
- 🚀 **CUDA** — GPU parallel merge (one thread per vocabulary word)

### Metrics calculated
| Metric | Formula | Meaning |
|--------|---------|----------|
| **Speedup** | T_seq / T_cuda | How many times faster CUDA is |
| **Parallel Efficiency** | Speedup / N_threads × 100% | How well each thread is used |
| **Throughput** | merges / second | Work done per unit time |
| **Time breakdown** | H2D + kernel + D2H | Where CUDA time is spent |

---
> ⚠️ **Before running:** `Runtime → Change runtime type → T4 GPU → Save`

---
## Step 1 — Verify GPU

In [ ]:
!nvidia-smi
print()
!nvcc --version

---
## Step 2 — Download Penn Treebank Dataset

In [ ]:
import urllib.request, os

BASE  = 'https://raw.githubusercontent.com/wojzaremba/lstm/master/data/'
FILES = ['ptb.train.txt', 'ptb.valid.txt', 'ptb.test.txt']

for fname in FILES:
    if not os.path.isfile(fname):
        print(f'Downloading {fname} ...')
        urllib.request.urlretrieve(BASE + fname, fname)
    kb = os.path.getsize(fname) / 1024
    print(f'  {fname:22s}  {kb:7.1f} KB')

print('\n✅  PTB files ready.')

---
## Step 3 — Write the CUDA Source File

The program runs **both** sequential and CUDA merges on identical copies of the vocabulary, then logs every timing measurement.

In [ ]:
cuda_source = r"""
/*
 * BPE Tokenizer — Sequential vs CUDA Efficiency Measurement
 * Penn Treebank Dataset
 * Compile: nvcc -O2 -o bpe_efficiency bpe_efficiency.cu
 */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>
#include <time.h>
#include <cuda_runtime.h>

#define HASH_SIZE       (1 << 17)
#define PAIR_HASH_SIZE  (1 << 18)
#define MAX_WORD_LEN    128
#define MAX_SYMBOLS     256
#define MAX_SYMBOL_LEN  64
#define MAX_VOCAB       50000
#define EOW             "</w>"
#define CUDA_BLOCK      256

static double now_sec(void) {
    struct timespec ts;
    clock_gettime(CLOCK_MONOTONIC, &ts);
    return ts.tv_sec + ts.tv_nsec * 1e-9;
}

typedef struct { int iter; double t_seq; double t_cuda; long pair_count;
                 char left[MAX_SYMBOL_LEN]; char right[MAX_SYMBOL_LEN]; } MergeTimingEntry;

typedef struct WordEntry {
    char word[MAX_WORD_LEN]; int freq;
    char syms[MAX_SYMBOLS][MAX_SYMBOL_LEN]; int nsyms;
    struct WordEntry *next;
} WordEntry;

typedef struct PairEntry {
    char left[MAX_SYMBOL_LEN]; char right[MAX_SYMBOL_LEN];
    long count; struct PairEntry *next;
} PairEntry;

static WordEntry *word_table_seq[HASH_SIZE];
static WordEntry *word_table_gpu[HASH_SIZE];
static PairEntry *pair_table[PAIR_HASH_SIZE];
static int        total_words = 0;

static unsigned int hash_str(const char *s, unsigned int mod) {
    unsigned int h = 5381;
    while (*s) h = h*31+(unsigned char)*s++; return h&(mod-1);
}
static unsigned int hash_pair(const char *a,const char *b,unsigned int mod){
    unsigned int h=5381;
    while(*a)h=h*31+(unsigned char)*a++; h=h*31+'#';
    while(*b)h=h*31+(unsigned char)*b++; return h&(mod-1);
}

static WordEntry *wfind(WordEntry **tbl,const char *w){
    for(WordEntry *e=tbl[hash_str(w,HASH_SIZE)];e;e=e->next)
        if(!strcmp(e->word,w))return e;
    return NULL;
}
static void winsert(WordEntry **tbl,const char *w,int freq){
    WordEntry *e=(WordEntry*)calloc(1,sizeof(WordEntry));
    strncpy(e->word,w,MAX_WORD_LEN-1); e->freq=freq;
    unsigned int idx=hash_str(w,HASH_SIZE); e->next=tbl[idx]; tbl[idx]=e;
}
static void copy_word_table(WordEntry **dst,WordEntry **src){
    for(int i=0;i<HASH_SIZE;i++){
        for(WordEntry *e=src[i];e;e=e->next){
            WordEntry *n=(WordEntry*)malloc(sizeof(WordEntry));
            memcpy(n,e,sizeof(WordEntry)); n->next=dst[i]; dst[i]=n;
        }
    }
}

static void pair_table_clear(void){
    for(int i=0;i<PAIR_HASH_SIZE;i++){
        PairEntry *e=pair_table[i];
        while(e){PairEntry *nx=e->next;free(e);e=nx;} pair_table[i]=NULL;
    }
}
static void pair_add(const char *l,const char *r,long cnt){
    unsigned int idx=hash_pair(l,r,PAIR_HASH_SIZE);
    for(PairEntry *e=pair_table[idx];e;e=e->next)
        if(!strcmp(e->left,l)&&!strcmp(e->right,r)){e->count+=cnt;return;}
    PairEntry *e=(PairEntry*)calloc(1,sizeof(PairEntry));
    strncpy(e->left,l,MAX_SYMBOL_LEN-1); strncpy(e->right,r,MAX_SYMBOL_LEN-1);
    e->count=cnt; e->next=pair_table[idx]; pair_table[idx]=e;
}
static void count_pairs_from(WordEntry **tbl){
    pair_table_clear();
    for(int i=0;i<HASH_SIZE;i++)
        for(WordEntry *e=tbl[i];e;e=e->next)
            for(int j=0;j+1<e->nsyms;j++)
                pair_add(e->syms[j],e->syms[j+1],(long)e->freq);
}
static PairEntry *find_best_pair(void){
    PairEntry *best=NULL; long bc=0;
    for(int i=0;i<PAIR_HASH_SIZE;i++)
        for(PairEntry *e=pair_table[i];e;e=e->next)
            if(e->count>bc){bc=e->count;best=e;}
    return best;
}

/* ── Sequential merge ── */
static void merge_seq(WordEntry **tbl,const char *left,const char *right){
    char merged[MAX_SYMBOL_LEN];
    snprintf(merged,MAX_SYMBOL_LEN,"%s%s",left,right);
    for(int i=0;i<HASH_SIZE;i++){
        for(WordEntry *e=tbl[i];e;e=e->next){
            int j=0;
            while(j<e->nsyms-1){
                if(!strcmp(e->syms[j],left)&&!strcmp(e->syms[j+1],right)){
                    strncpy(e->syms[j],merged,MAX_SYMBOL_LEN-1);
                    for(int k=j+1;k<e->nsyms-1;k++)
                        memcpy(e->syms[k],e->syms[k+1],MAX_SYMBOL_LEN);
                    e->nsyms--;
                } else j++;
            }
        }
    }
}

/* ── CUDA kernel ── */
__global__ void merge_kernel(char *d_syms,int *d_nsyms,int nwords,
                              const char *d_left,const char *d_right,char *d_merged){
    int wid=blockIdx.x*blockDim.x+threadIdx.x;
    if(wid>=nwords)return;
    char (*syms)[MAX_SYMBOL_LEN]=(char(*)[MAX_SYMBOL_LEN])
        (d_syms+(long)wid*MAX_SYMBOLS*MAX_SYMBOL_LEN);
    int nsyms=d_nsyms[wid],i=0;
    while(i<nsyms-1){
        bool ml=true;
        for(int k=0;k<MAX_SYMBOL_LEN&&(d_left[k]||syms[i][k]);k++)
            if(syms[i][k]!=d_left[k]){ml=false;break;}
        if(!ml){i++;continue;}
        bool mr=true;
        for(int k=0;k<MAX_SYMBOL_LEN&&(d_right[k]||syms[i+1][k]);k++)
            if(syms[i+1][k]!=d_right[k]){mr=false;break;}
        if(!mr){i++;continue;}
        for(int k=0;k<MAX_SYMBOL_LEN;k++)syms[i][k]=d_merged[k];
        for(int j=i+1;j<nsyms-1;j++)
            for(int k=0;k<MAX_SYMBOL_LEN;k++)syms[j][k]=syms[j+1][k];
        nsyms--;
    }
    d_nsyms[wid]=nsyms;
}

static WordEntry **wlist_gpu=NULL; static int wlist_gpu_n=0;
static void build_flat_list(WordEntry **tbl){
    free(wlist_gpu);
    wlist_gpu=(WordEntry**)malloc(total_words*sizeof(WordEntry*));
    wlist_gpu_n=0;
    for(int i=0;i<HASH_SIZE;i++)
        for(WordEntry *e=tbl[i];e;e=e->next)
            wlist_gpu[wlist_gpu_n++]=e;
}

static double merge_cuda(const char *left,const char *right,
                         double *t_h2d,double *t_kern,double *t_d2h){
    int N=wlist_gpu_n;
    size_t sb=(size_t)N*MAX_SYMBOLS*MAX_SYMBOL_LEN;
    char *hs=(char*)malloc(sb); int *hn=(int*)malloc(N*sizeof(int));
    for(int i=0;i<N;i++){
        WordEntry *e=wlist_gpu[i]; hn[i]=e->nsyms;
        char(*dst)[MAX_SYMBOL_LEN]=(char(*)[MAX_SYMBOL_LEN])(hs+(long)i*MAX_SYMBOLS*MAX_SYMBOL_LEN);
        for(int j=0;j<e->nsyms;j++)memcpy(dst[j],e->syms[j],MAX_SYMBOL_LEN);
    }
    char merged[MAX_SYMBOL_LEN]; snprintf(merged,MAX_SYMBOL_LEN,"%s%s",left,right);
    char *ds,*dl,*dr,*dm; int *dn;
    cudaMalloc(&ds,sb); cudaMalloc(&dn,N*sizeof(int));
    cudaMalloc(&dl,MAX_SYMBOL_LEN); cudaMalloc(&dr,MAX_SYMBOL_LEN);
    cudaMalloc(&dm,MAX_SYMBOL_LEN);
    double t0=now_sec();
    cudaMemcpy(ds,hs,sb,cudaMemcpyHostToDevice);
    cudaMemcpy(dn,hn,N*sizeof(int),cudaMemcpyHostToDevice);
    cudaMemcpy(dl,left,MAX_SYMBOL_LEN,cudaMemcpyHostToDevice);
    cudaMemcpy(dr,right,MAX_SYMBOL_LEN,cudaMemcpyHostToDevice);
    cudaMemcpy(dm,merged,MAX_SYMBOL_LEN,cudaMemcpyHostToDevice);
    double t1=now_sec();
    int blk=(N+CUDA_BLOCK-1)/CUDA_BLOCK;
    merge_kernel<<<blk,CUDA_BLOCK>>>(ds,dn,N,dl,dr,dm);
    cudaDeviceSynchronize();
    double t2=now_sec();
    cudaMemcpy(hs,ds,sb,cudaMemcpyDeviceToHost);
    cudaMemcpy(hn,dn,N*sizeof(int),cudaMemcpyDeviceToHost);
    double t3=now_sec();
    for(int i=0;i<N;i++){
        WordEntry *e=wlist_gpu[i]; e->nsyms=hn[i];
        char(*src)[MAX_SYMBOL_LEN]=(char(*)[MAX_SYMBOL_LEN])(hs+(long)i*MAX_SYMBOLS*MAX_SYMBOL_LEN);
        for(int j=0;j<e->nsyms;j++)memcpy(e->syms[j],src[j],MAX_SYMBOL_LEN);
    }
    cudaFree(ds);cudaFree(dn);cudaFree(dl);cudaFree(dr);cudaFree(dm);
    free(hs);free(hn);
    if(t_h2d)*t_h2d=t1-t0;
    if(t_kern)*t_kern=t2-t1;
    if(t_d2h)*t_d2h=t3-t2;
    return t3-t0;
}

static void init_symbols(WordEntry **tbl){
    for(int i=0;i<HASH_SIZE;i++){
        for(WordEntry *e=tbl[i];e;e=e->next){
            e->nsyms=0; const char *p=e->word;
            while(*p&&e->nsyms<MAX_SYMBOLS-1){
                e->syms[e->nsyms][0]=*p++; e->syms[e->nsyms][1]='\0'; e->nsyms++;}
            strncpy(e->syms[e->nsyms++],EOW,MAX_SYMBOL_LEN-1);
        }
    }
}

static void read_text(const char *filename,WordEntry **tbl){
    FILE *f=fopen(filename,"r"); if(!f){perror(filename);exit(1);}
    WordEntry *tmp[HASH_SIZE]; memset(tmp,0,sizeof(tmp));
    char buf[MAX_WORD_LEN]; int idx=0,c;
    while((c=fgetc(f))!=EOF){
        if(isalpha(c)||c=='<'||c=='>'||c=='_'||c=='-'){
            if(idx<MAX_WORD_LEN-1)buf[idx++]=(char)tolower(c);
        } else {
            if(idx>0){buf[idx]='\0';
                WordEntry *e=wfind(tmp,buf);
                if(e)e->freq++; else{winsert(tmp,buf,1);total_words++;}
                idx=0;}
        }
    }
    if(idx>0){buf[idx]='\0';
        WordEntry *e=wfind(tmp,buf);
        if(e)e->freq++; else{winsert(tmp,buf,1);total_words++;}
    }
    fclose(f);
    for(int i=0;i<HASH_SIZE;i++){
        for(WordEntry *e=tmp[i];e;e=e->next)winsert(tbl,e->word,e->freq);
        WordEntry *e=tmp[i]; while(e){WordEntry *nx=e->next;free(e);e=nx;}
    }
}

int main(int argc,char **argv){
    if(argc<3){fprintf(stderr,"Usage: %s <file> <merges>\n",argv[0]);return 1;}
    int num_merges=atoi(argv[2]);
    int dev; cudaGetDevice(&dev);
    cudaDeviceProp prop; cudaGetDeviceProperties(&prop,dev);
    printf("GPU: %s  SMs:%d  VRAM:%.0fMB\n",prop.name,prop.multiProcessorCount,prop.totalGlobalMem/1e6);
    printf("Reading %s ...\n",argv[1]);
    read_text(argv[1],word_table_seq);
    printf("Unique words: %d\n",total_words);
    copy_word_table(word_table_gpu,word_table_seq);
    init_symbols(word_table_seq); init_symbols(word_table_gpu);
    build_flat_list(word_table_gpu);
    int gpu_threads=((total_words+CUDA_BLOCK-1)/CUDA_BLOCK)*CUDA_BLOCK;
    MergeTimingEntry *log=(MergeTimingEntry*)malloc(num_merges*sizeof(MergeTimingEntry));
    double total_seq=0,total_cuda=0; int actual_n=0;
    printf("\n%6s  %-14s  %10s  %10s  %8s  %10s  %10s\n",
           "Iter","BestPair","Seq(ms)","CUDA(ms)","Speedup","H2D(ms)","Kern(ms)");
    printf("%.85s\n","-------------------------------------------------------------------------------------");
    for(int iter=0;iter<num_merges;iter++){
        count_pairs_from(word_table_seq);
        PairEntry *best=find_best_pair();
        if(!best||best->count<2){printf("No more pairs at iter %d\n",iter);break;}
        char left[MAX_SYMBOL_LEN],right[MAX_SYMBOL_LEN];
        strncpy(left,best->left,MAX_SYMBOL_LEN-1);
        strncpy(right,best->right,MAX_SYMBOL_LEN-1);
        long pc=best->count;
        double ts0=now_sec(); merge_seq(word_table_seq,left,right); double t_seq=now_sec()-ts0;
        double t_h2d,t_kern,t_d2h;
        double t_cuda=merge_cuda(left,right,&t_h2d,&t_kern,&t_d2h);
        total_seq+=t_seq; total_cuda+=t_cuda;
        log[actual_n].iter=iter; log[actual_n].t_seq=t_seq; log[actual_n].t_cuda=t_cuda;
        log[actual_n].pair_count=pc;
        strncpy(log[actual_n].left,left,MAX_SYMBOL_LEN-1);
        strncpy(log[actual_n].right,right,MAX_SYMBOL_LEN-1);
        actual_n++;
        if(iter<20||iter%100==0){
            char pair[32]; snprintf(pair,32,"%s+%s",left,right);
            double sp=(t_cuda>0)?t_seq/t_cuda:0;
            printf("%6d  %-14s  %10.4f  %10.4f  %8.2f  %10.4f  %10.4f\n",
                   iter,pair,t_seq*1000,t_cuda*1000,sp,t_h2d*1000,t_kern*1000);
        }
    }
    double speedup=total_seq/total_cuda;
    double efficiency=speedup/gpu_threads*100.0;
    printf("\n╔══════════════════════════════════════════════════════╗\n");
    printf("║              EFFICIENCY REPORT                       ║\n");
    printf("╠══════════════════════════════════════════════════════╣\n");
    printf("║  GPU                  : %-28s║\n",prop.name);
    printf("║  Merge iterations     : %-28d║\n",actual_n);
    printf("║  Unique vocab words   : %-28d║\n",total_words);
    printf("║  GPU threads launched : %-28d║\n",gpu_threads);
    printf("╠══════════════════════════════════════════════════════╣\n");
    printf("║  Sequential total     : %-.4f s                  ║\n",total_seq);
    printf("║  CUDA total           : %-.4f s                  ║\n",total_cuda);
    printf("║  Speedup (T_seq/T_gpu): %.2f x                      ║\n",speedup);
    printf("║  Parallel Efficiency  : %.6f %%                ║\n",efficiency);
    printf("║  Seq  throughput      : %.1f merges/s            ║\n",actual_n/total_seq);
    printf("║  CUDA throughput      : %.1f merges/s            ║\n",actual_n/total_cuda);
    printf("╚══════════════════════════════════════════════════════╝\n");
    FILE *fc=fopen("efficiency_log.csv","w");
    if(fc){
        fprintf(fc,"iter,left,right,pair_count,t_seq_ms,t_cuda_ms,speedup\n");
        for(int i=0;i<actual_n;i++){
            double sp=(log[i].t_cuda>0)?log[i].t_seq/log[i].t_cuda:0;
            fprintf(fc,"%d,%s,%s,%ld,%.6f,%.6f,%.4f\n",
                    log[i].iter,log[i].left,log[i].right,log[i].pair_count,
                    log[i].t_seq*1000,log[i].t_cuda*1000,sp);
        }
        fclose(fc); printf("CSV -> efficiency_log.csv\n");
    }
    FILE *fj=fopen("efficiency_summary.json","w");
    if(fj){
        fprintf(fj,"{\n  \"gpu\": \"%s\",\n  \"num_merges\": %d,\n  \"unique_words\": %d,\n"
                   "  \"gpu_threads\": %d,\n  \"total_seq_s\": %.6f,\n  \"total_cuda_s\": %.6f,\n"
                   "  \"speedup\": %.4f,\n  \"parallel_efficiency_pct\": %.6f,\n"
                   "  \"seq_throughput\": %.2f,\n  \"cuda_throughput\": %.2f\n}\n",
                prop.name,actual_n,total_words,gpu_threads,
                total_seq,total_cuda,speedup,efficiency,
                actual_n/total_seq,actual_n/total_cuda);
        fclose(fj); printf("JSON -> efficiency_summary.json\n");
    }
    free(log); free(wlist_gpu);
    return 0;
}
"""

with open('bpe_efficiency.cu', 'w') as f:
    f.write(cuda_source)
print(f'✅  bpe_efficiency.cu written  ({cuda_source.count(chr(10))} lines)')

---
## Step 4 — Compile

In [ ]:
!nvcc -O2 -o bpe_efficiency bpe_efficiency.cu 2>&1
import os
if os.path.isfile('bpe_efficiency'):
    print('\n✅  Compilation successful!')
else:
    print('\n❌  Compilation FAILED — read errors above')

---
## Step 5 — Run Efficiency Benchmark

Set `NUM_MERGES`. Both sequential and CUDA merges run on **identical** vocabulary copies so timings are directly comparable.

In [ ]:
import time
NUM_MERGES = 1000   # ← change this (500=fast demo, 2000=full benchmark)

print(f'Running {NUM_MERGES} BPE iterations measuring Sequential vs CUDA ...')
print('=' * 80)
t0 = time.time()
!./bpe_efficiency ptb.train.txt {NUM_MERGES}
print('=' * 80)
print(f'Total wall clock: {time.time()-t0:.1f} s')

---
## Step 6 — Load & Display Results

In [ ]:
import pandas as pd, json

# load CSV log
df = pd.read_csv('efficiency_log.csv')
print(f'Loaded {len(df)} rows from efficiency_log.csv')
print(df.head(10).to_string(index=False))

# load JSON summary
with open('efficiency_summary.json') as f:
    summary = json.load(f)

print('\n--- EFFICIENCY SUMMARY ---')
for k, v in summary.items():
    print(f'  {k:35s}: {v}')

---
## Step 7 — Visualise Efficiency Metrics

Five charts showing every measured dimension of performance.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

df['speedup'] = df['t_seq_ms'] / df['t_cuda_ms'].replace(0, float('nan'))

fig, axes = plt.subplots(3, 2, figsize=(15, 15))
fig.suptitle(
    f'BPE Tokenizer — Sequential vs CUDA Efficiency\n'
    f'Penn Treebank  |  {len(df)} merge iterations  |  GPU: {summary["gpu"]}',
    fontsize=14, fontweight='bold', y=1.01
)

# ── Chart 1: Time per merge iteration ──────────────────────────
ax = axes[0, 0]
ax.plot(df['iter'], df['t_seq_ms'],  label='Sequential (CPU)', color='tomato',   lw=1.2, alpha=0.9)
ax.plot(df['iter'], df['t_cuda_ms'], label='CUDA (GPU)',        color='steelblue',lw=1.2, alpha=0.9)
ax.set_xlabel('Merge Iteration'); ax.set_ylabel('Time (ms)')
ax.set_title('Time per Merge Iteration')
ax.legend(); ax.grid(alpha=0.3)

# ── Chart 2: Speedup over iterations ───────────────────────────
ax = axes[0, 1]
ax.plot(df['iter'], df['speedup'], color='darkorange', lw=1.4)
ax.axhline(df['speedup'].median(), color='gray', ls='--', lw=1,
           label=f'Median speedup = {df["speedup"].median():.2f}x')
ax.set_xlabel('Merge Iteration'); ax.set_ylabel('Speedup (T_seq / T_cuda)')
ax.set_title('Speedup per Iteration')
ax.legend(); ax.grid(alpha=0.3)

# ── Chart 3: Cumulative time comparison ────────────────────────
ax = axes[1, 0]
ax.plot(df['iter'], df['t_seq_ms'].cumsum(),  label='Sequential cumulative', color='tomato',    lw=1.5)
ax.plot(df['iter'], df['t_cuda_ms'].cumsum(), label='CUDA cumulative',        color='steelblue', lw=1.5)
ax.set_xlabel('Merge Iteration'); ax.set_ylabel('Cumulative Time (ms)')
ax.set_title('Cumulative Time — Sequential vs CUDA')
ax.legend(); ax.grid(alpha=0.3)

# ── Chart 4: Pair count over iterations (corpus coverage) ──────
ax = axes[1, 1]
ax.bar(df['iter'], df['pair_count'], color='mediumseagreen', alpha=0.7, width=1)
ax.set_xlabel('Merge Iteration'); ax.set_ylabel('Best Pair Frequency')
ax.set_title('Best Pair Count per Iteration\n(shows how merges become rarer over time)')
ax.grid(alpha=0.3, axis='y')

# ── Chart 5: Rolling average speedup ───────────────────────────
ax = axes[2, 0]
window = max(1, len(df) // 20)
roll_sp = df['speedup'].rolling(window, min_periods=1).mean()
ax.plot(df['iter'], roll_sp, color='purple', lw=1.8,
        label=f'Rolling avg speedup (window={window})')
ax.axhline(1.0, color='gray', ls=':', lw=1, label='Baseline (speedup=1)')
ax.set_xlabel('Merge Iteration'); ax.set_ylabel('Speedup')
ax.set_title('Rolling Average Speedup')
ax.legend(); ax.grid(alpha=0.3)

# ── Chart 6: Summary bar chart ─────────────────────────────────
ax = axes[2, 1]
metrics = ['Total\nSeq Time (s)', 'Total\nCUDA Time (s)',
           'Speedup (x)', 'CUDA Throughput\n(merges/s ÷ 10)']
values  = [
    summary['total_seq_s'],
    summary['total_cuda_s'],
    summary['speedup'],
    summary['cuda_throughput'] / 10
]
colors = ['tomato', 'steelblue', 'darkorange', 'mediumseagreen']
bars = ax.bar(metrics, values, color=colors, alpha=0.85, edgecolor='white', linewidth=1.2)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title('Summary Metrics')
ax.set_ylabel('Value'); ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('efficiency_charts.png', dpi=130, bbox_inches='tight')
plt.show()
print('✅  Charts saved as efficiency_charts.png')

---
## Step 8 — Full Efficiency Table

In [ ]:
import json
with open('efficiency_summary.json') as f:
    s = json.load(f)

print('╔' + '═'*62 + '╗')
print('║{:^62}║'.format('  COMPLETE EFFICIENCY REPORT  '))
print('╠' + '═'*62 + '╣')
rows = [
    ('GPU',                      s['gpu']),
    ('Merge Iterations',         s['num_merges']),
    ('Unique Vocab Words',        s['unique_words']),
    ('GPU Threads Launched',      s['gpu_threads']),
    ('',                          ''),
    ('Sequential Total Time',     f"{s['total_seq_s']:.4f} s"),
    ('CUDA Total Time',           f"{s['total_cuda_s']:.4f} s"),
    ('',                          ''),
    ('Speedup  (T_seq / T_cuda)', f"{s['speedup']:.4f} x"),
    ('Parallel Efficiency',       f"{s['parallel_efficiency_pct']:.8f} %"),
    ('Sequential Throughput',     f"{s['seq_throughput']:.1f} merges/s"),
    ('CUDA Throughput',           f"{s['cuda_throughput']:.1f} merges/s"),
    ('Throughput Gain',           f"{s['cuda_throughput']/s['seq_throughput']:.2f} x"),
]
for k, v in rows:
    if k == '':
        print('╟' + '─'*62 + '╢')
    else:
        line = f'  {k:<30}  {str(v):<28}'
        print(f'║{line}║')
print('╚' + '═'*62 + '╝')

print('\n📌 Interpretation:')
sp = s['speedup']
eff = s['parallel_efficiency_pct']
if sp > 1:
    print(f'  ✅  CUDA is {sp:.2f}x FASTER than sequential.')
else:
    print(f'  ⚠️   Sequential is faster here — CUDA overhead dominates at small vocab.')
print(f'  📊  Parallel efficiency = {eff:.4f}%')
print(f'      (Each GPU thread is contributing {eff:.4f}% of ideal linear scaling)')
print(f'  💡  Low efficiency is expected: PCIe transfer overhead per merge iteration')
print(f'      dominates at small vocabulary sizes. Efficiency improves with larger corpora.')

---
## Step 9 — Download All Results

In [ ]:
from google.colab import files
for fname in ['efficiency_log.csv', 'efficiency_summary.json', 'efficiency_charts.png']:
    if os.path.isfile(fname):
        files.download(fname)
        print(f'⬇  {fname}')

---
## Efficiency Formula Reference

```
┌────────────────────────────────────────────────────────────────┐
│  METRIC              FORMULA                  MEANING          │
├────────────────────────────────────────────────────────────────┤
│  Speedup S           T_seq / T_cuda           How much faster  │
│  Parallel Efficiency S / N_threads × 100%     Thread usage %   │
│  Throughput (seq)    N_merges / T_seq          merges/second    │
│  Throughput (CUDA)   N_merges / T_cuda         merges/second    │
│  H2D overhead        t_memcpy_host_to_device   Transfer cost    │
│  Kernel time         t_cuda_kernel_only         Pure GPU work   │
│  D2H overhead        t_memcpy_device_to_host   Transfer cost    │
└────────────────────────────────────────────────────────────────┘

Why parallel efficiency is small:
  - PCIe bandwidth is the bottleneck (data must move each iteration)
  - PTB vocabulary (~10K words) fits fewer blocks than a full GPU needs
  - At larger scale (Wikipedia, 200K+ words) efficiency rises significantly

CUDA parallelism model:
  vocabulary_words = N
  blocks           = ceil(N / 256)
  threads_per_block = 256
  → all N words merge simultaneously in one kernel call
```